In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import train_test_split

# Label mappings — required to decode model output back to string labels
domain2id    = {'nfl': 0, 'financial': 1, 'politics': 2, 'videogames': 3, 'none': 4}
id2domain    = {v: k for k, v in domain2id.items()}
sentiment2id = {'negative': 0, 'positive': 1, "neutral": 2}
id2sentiment = {v: k for k, v in sentiment2id.items()}

# Recreate the architecture 
# Must match the original definition exactly for load_state_dict to work
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, num_domains, num_sentiments, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size

        # Separate classification heads for each task
        self.domain_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, num_domains)
        )
        self.sentiment_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, num_sentiments)
        )

    def forward(self, input_ids, attention_mask, labels=None, sentiment_labels=None):
        outputs   = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        # Use the [CLS] token as the sentence-level representation
        cls_token = outputs.last_hidden_state[:, 0, :]

        domain_logits    = self.domain_head(cls_token)
        sentiment_logits = self.sentiment_head(cls_token)

        # Combine both task losses equally when labels are provided
        loss = None
        if labels is not None and sentiment_labels is not None:
            loss = (
                torch.nn.functional.cross_entropy(domain_logits, labels) +
                torch.nn.functional.cross_entropy(sentiment_logits, sentiment_labels)
            )

        return {
            'loss':             loss,
            'domain_logits':    domain_logits,
            'sentiment_logits': sentiment_logits,
        }

# Device 
# Prefer MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

#  Load weights 
# Rebuild the model architecture, then load the saved weights into it
backbone = AutoModel.from_pretrained('distilbert-base-uncased')
model    = MultiTaskModel(backbone, num_domains=5, num_sentiments=3)
model.load_state_dict(torch.load('./multitask-classifier/model_weights.pt',
                                  map_location=device, weights_only=True))
model.to(device).eval()

# Load the tokenizer saved alongside the model weights
tokenizer = AutoTokenizer.from_pretrained('./multitask-classifier')
print("Model loaded and ready")

Using device: mps
Model loaded and ready


To access model weights, please refer to this link: https://drive.google.com/drive/folders/1OBvHxZj7SinyVyKf3bKt-fu_ZPCQDDmh?usp=sharing

In [2]:
import pandas as pd
import torch.nn.functional as F
from sklearn.metrics import classification_report

# Load the videogames dataset and prepare it for evaluation
videogames_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/videogames_train.csv")

# Normalize sentiment labels to lowercase for consistency
videogames_df['sentiment'] = videogames_df['sentiment'].str.lower()
videogames_df['domain']    = 'videogames'

# Split into train/test sets (80/20), stratified to preserve sentiment distribution
train_vg_df, test_vg_df = train_test_split(
    videogames_df[['text', 'sentiment', 'domain']],
    test_size=0.2,
    random_state=42,
    stratify=videogames_df['sentiment']
)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# Batch predict 
def batch_predict(texts, batch_size=64):
    all_results = []
    # Process in batches to avoid OOM on large inputs
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        for j in range(len(batch)):
            # Use only the sentiment head output for this evaluation
            sentiment_probs = F.softmax(outputs['sentiment_logits'][j], dim=-1)
            all_results.append({
                'text':            batch[j],
                'sentiment':       id2sentiment[sentiment_probs.argmax().item()],
                'sentiment_score': round(sentiment_probs.max().item(), 3),
            })

        # Log progress every 20 batches to avoid flooding stdout
        if (i // batch_size) % 20 == 0:
            print(f"  Processed {min(i + batch_size, len(texts)):,} / {len(texts):,}", end='\r')

    print(f"  Done — processed {len(texts):,} texts")
    return all_results

# Evaluate 
texts                 = test_vg_df['text'].astype(str).tolist()
# Map string sentiment labels to integer IDs for comparison with predictions
true_sentiment_labels = test_vg_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (videogames) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 7,618 texts

=== Sentiment Classification Report (videogames) ===
              precision    recall  f1-score   support

    negative       0.71      0.80      0.75      2776
    positive       0.72      0.74      0.73      2858
     neutral       0.60      0.47      0.52      1984

    accuracy                           0.69      7618
   macro avg       0.67      0.67      0.67      7618
weighted avg       0.68      0.69      0.68      7618



In [4]:
# Classify sentiment based on score thresholds
def classify_sentiment(score):
    if score < -0.1:
        return 'negative'
    elif score > 0.1:
        return 'positive'
    else:
        return 'neutral'

# Load the financial tweets dataset
financial_df = pd.read_csv('/Users/alecxszhang/Desktop/Stat 359/data/Financial_tweets.csv')

# Apply sentiment classification and tag the domain
financial_df['sentiment'] = financial_df['score'].apply(classify_sentiment)
financial_df['domain']    = 'financial'

# Standardize text column name for consistency across datasets
financial_df = financial_df.rename(columns={'full_text': 'text'})

# Split into train/test sets (80/20), stratified to preserve sentiment distribution
train_fin_df, test_fin_df = train_test_split(
    financial_df[['text', 'sentiment', 'domain']],
    test_size=0.2,
    random_state=42,
    stratify=financial_df['sentiment']
)

In [5]:
# Evaluate sentiment predictions on the financial test set
texts                 = test_fin_df['text'].astype(str).tolist()
# Map string sentiment labels to integer IDs for comparison with predictions
true_sentiment_labels = test_fin_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (Financial) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 2,484 texts

=== Sentiment Classification Report (Financial) ===
              precision    recall  f1-score   support

    negative       0.76      0.67      0.71       784
    positive       0.71      0.53      0.61       808
     neutral       0.56      0.75      0.64       892

    accuracy                           0.65      2484
   macro avg       0.68      0.65      0.66      2484
weighted avg       0.67      0.65      0.66      2484



In [6]:
# Load the politics dataset
politics_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/politics_sentiment.csv")

# Tag the domain and standardize text column name for consistency across datasets
politics_df['domain'] = 'politics'
politics_df = politics_df.rename(columns={'tweet': 'text'})

# Map numeric sentiment labels to string equivalents
politics_df['sentiment'] = politics_df['sentiment'].map({0: 'negative', 1: 'positive', 2: 'neutral'})

# Drop rows where sentiment mapping failed (unmapped/unexpected values)
politics_df = politics_df.dropna(subset=['sentiment'])

# Split into train/test sets (80/20), stratified to preserve sentiment distribution
train_pol_df, test_pol_df = train_test_split(
    politics_df[['text', 'sentiment', 'domain']],
    test_size=0.2,
    random_state=42,
    stratify=politics_df['sentiment']
)

In [7]:
# Evaluate sentiment predictions on the politics test set
texts                 = test_pol_df['text'].astype(str).tolist()
# Map string sentiment labels to integer IDs for comparison with predictions
true_sentiment_labels = test_pol_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (Politics) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 1,091 texts

=== Sentiment Classification Report (Politics) ===
              precision    recall  f1-score   support

    negative       0.77      0.80      0.79       496
    positive       0.79      0.76      0.77       484
     neutral       0.82      0.81      0.81       111

    accuracy                           0.78      1091
   macro avg       0.79      0.79      0.79      1091
weighted avg       0.78      0.78      0.78      1091



In [8]:
# Load the NFL dataset
nfl_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/nfl_sentiments.csv")

# Tag the domain
nfl_df['domain'] = 'nfl'

# Filter to only tweets that mention NFL-related keywords
# to remove off-topic scraped content
import re
nfl_keywords = r'\b(nfl|football|touchdown|quarterback|qb|mahomes|brady|receiver|linebacker|' \
               r'packers|chiefs|eagles|cowboys|49ers|ravens|browns|patriots|bills|' \
               r'draft|super bowl|playoffs|offense|defense|interception|fumble|yard|'\
               r'rushing|passing|receiver|tight end|kicker)\b'

nfl_df = nfl_df[nfl_df['text'].str.contains(nfl_keywords, case=False, regex=True)]

# Keep only rows with valid sentiment labels (drop any unexpected values)
nfl_df = nfl_df[nfl_df['sentiment'].isin(['negative', 'positive', 'neutral'])]

# Split into train/test sets (80/20), stratified to preserve sentiment distribution
train_nfl_df, test_nfl_df = train_test_split(
    nfl_df[['text', 'sentiment', 'domain']],
    test_size=0.2,
    random_state=42,
    stratify=nfl_df['sentiment']
)

/var/folders/2p/rhd39m592jv2kqghpzh8s5r40000gp/T/ipykernel_1001/1969130544.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nfl_df = nfl_df[nfl_df['text'].str.contains(nfl_keywords, case=False, regex=True)]


In [9]:
# Evaluate sentiment predictions on the NFL test set
texts                 = test_nfl_df['text'].astype(str).tolist()
# Map string sentiment labels to integer IDs for comparison with predictions
true_sentiment_labels = test_nfl_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (NFL) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 564 texts

=== Sentiment Classification Report (NFL) ===
              precision    recall  f1-score   support

    negative       0.80      0.77      0.78       189
    positive       0.75      0.72      0.74       119
     neutral       0.79      0.83      0.81       256

    accuracy                           0.79       564
   macro avg       0.78      0.77      0.78       564
weighted avg       0.79      0.79      0.78       564

